In [1]:
import pandas as pd

books = pd.read_csv("books_with_categories.csv")

In [8]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = -1)
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528684265911579},
  {'label': 'neutral', 'score': 0.005764580797404051},
  {'label': 'anger', 'score': 0.004419785924255848},
  {'label': 'sadness', 'score': 0.002092391485348344},
  {'label': 'disgust', 'score': 0.001611991785466671},
  {'label': 'fear', 'score': 0.0004138525982853025}]]

In [4]:
books["description"][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [9]:
classifier(books["description"][0])

[[{'label': 'fear', 'score': 0.6548401713371277},
  {'label': 'neutral', 'score': 0.16985252499580383},
  {'label': 'sadness', 'score': 0.11640933156013489},
  {'label': 'surprise', 'score': 0.020700689405202866},
  {'label': 'disgust', 'score': 0.01910070702433586},
  {'label': 'joy', 'score': 0.015161466784775257},
  {'label': 'anger', 'score': 0.0039351508021354675}]]

In [10]:
classifier(books["description"][0].split("."))

[[{'label': 'surprise', 'score': 0.729602038860321},
  {'label': 'neutral', 'score': 0.14038604497909546},
  {'label': 'fear', 'score': 0.06816232949495316},
  {'label': 'joy', 'score': 0.04794265329837799},
  {'label': 'anger', 'score': 0.009156365878880024},
  {'label': 'disgust', 'score': 0.002628476358950138},
  {'label': 'sadness', 'score': 0.0021221654023975134}],
 [{'label': 'neutral', 'score': 0.44937023520469666},
  {'label': 'disgust', 'score': 0.27359169721603394},
  {'label': 'joy', 'score': 0.10908326506614685},
  {'label': 'sadness', 'score': 0.09362727403640747},
  {'label': 'anger', 'score': 0.040478333830833435},
  {'label': 'surprise', 'score': 0.026970140635967255},
  {'label': 'fear', 'score': 0.006879040505737066}],
 [{'label': 'neutral', 'score': 0.6462150812149048},
  {'label': 'sadness', 'score': 0.24273402988910675},
  {'label': 'disgust', 'score': 0.04342275485396385},
  {'label': 'surprise', 'score': 0.028300579637289047},
  {'label': 'joy', 'score': 0.014211

In [11]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)

In [12]:
sentences[0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives'

In [13]:
predictions[0]

[{'label': 'surprise', 'score': 0.729602038860321},
 {'label': 'neutral', 'score': 0.14038604497909546},
 {'label': 'fear', 'score': 0.06816232949495316},
 {'label': 'joy', 'score': 0.04794265329837799},
 {'label': 'anger', 'score': 0.009156365878880024},
 {'label': 'disgust', 'score': 0.002628476358950138},
 {'label': 'sadness', 'score': 0.0021221654023975134}]

In [14]:
sentences[3]


' Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist'

In [15]:
predictions[3]


[{'label': 'fear', 'score': 0.9281682372093201},
 {'label': 'anger', 'score': 0.032190948724746704},
 {'label': 'neutral', 'score': 0.012808676809072495},
 {'label': 'sadness', 'score': 0.008756890892982483},
 {'label': 'surprise', 'score': 0.008597920648753643},
 {'label': 'disgust', 'score': 0.00843182671815157},
 {'label': 'joy', 'score': 0.0010455843294039369}]

In [16]:
sorted(predictions[0], key=lambda x: x["label"])

[{'label': 'anger', 'score': 0.009156365878880024},
 {'label': 'disgust', 'score': 0.002628476358950138},
 {'label': 'fear', 'score': 0.06816232949495316},
 {'label': 'joy', 'score': 0.04794265329837799},
 {'label': 'neutral', 'score': 0.14038604497909546},
 {'label': 'sadness', 'score': 0.0021221654023975134},
 {'label': 'surprise', 'score': 0.729602038860321}]

In [17]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label : [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}

    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [18]:
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [19]:
emotion_scores

{'anger': [np.float64(0.0641336441040039),
  np.float64(0.6126199960708618),
  np.float64(0.0641336441040039),
  np.float64(0.3514844477176666),
  np.float64(0.08141247928142548),
  np.float64(0.23222437500953674),
  np.float64(0.5381838083267212),
  np.float64(0.0641336441040039),
  np.float64(0.3006701171398163),
  np.float64(0.0641336441040039)],
 'disgust': [np.float64(0.27359169721603394),
  np.float64(0.3482848107814789),
  np.float64(0.10400670766830444),
  np.float64(0.15072225034236908),
  np.float64(0.18449555337429047),
  np.float64(0.727175235748291),
  np.float64(0.155854731798172),
  np.float64(0.10400670766830444),
  np.float64(0.2794811725616455),
  np.float64(0.1779259592294693)],
 'fear': [np.float64(0.9281682372093201),
  np.float64(0.9425276517868042),
  np.float64(0.9723207950592041),
  np.float64(0.360706090927124),
  np.float64(0.09504348784685135),
  np.float64(0.05136282369494438),
  np.float64(0.7474274635314941),
  np.float64(0.4044972062110901),
  np.float64

In [20]:
from tqdm import tqdm


emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label : [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])


  3%|▎         | 180/5197 [00:49<28:16,  2.96it/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]


100%|██████████| 5197/5197 [22:47<00:00,  3.80it/s]


In [21]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [22]:
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.729602,9780002005883
1,0.612620,0.348285,0.942528,0.704422,0.887940,0.111690,0.252547,9780002261982
2,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,0.078765,9780006178736
3,0.351484,0.150722,0.360706,0.251881,0.732685,0.111690,0.078765,9780006280897
4,0.081412,0.184496,0.095043,0.040564,0.884390,0.475880,0.078765,9780006280934
...,...,...,...,...,...,...,...,...
5192,0.148209,0.030643,0.919165,0.255172,0.853721,0.980877,0.030656,9788172235222
5193,0.064134,0.114383,0.051363,0.400263,0.883198,0.111690,0.227765,9788173031014
5194,0.009997,0.009929,0.339218,0.947779,0.375755,0.066685,0.057625,9788179921623
5195,0.064134,0.104007,0.459270,0.759457,0.951104,0.368111,0.078765,9788185300535


In [23]:
books = pd.merge(books, emotions_df, on = "isbn13" )

In [24]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.729602
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.612620,0.348285,0.942528,0.704422,0.887940,0.111690,0.252547
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,0.078765
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.351484,0.150722,0.360706,0.251881,0.732685,0.111690,0.078765
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.081412,0.184496,0.095043,0.040564,0.884390,0.475880,0.078765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,...,Mistaken Identity,9788172235222 On A Train Journey Home To North...,Fiction,0.148209,0.030643,0.919165,0.255172,0.853721,0.980877,0.030656
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,...,Journey to the East,9788173031014 This book tells the tale of a ma...,Nonfiction,0.064134,0.114383,0.051363,0.400263,0.883198,0.111690,0.227765
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,...,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction,0.009997,0.009929,0.339218,0.947779,0.375755,0.066685,0.057625
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,...,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction,0.064134,0.104007,0.459270,0.759457,0.951104,0.368111,0.078765


In [25]:
books.to_csv("books_with_emotions.csv", index = False)